# Paper SOTA comparison (IMC25 metric)

This notebook builds **paper-ready comparison tables** across multiple methods on a labeled split.

It:
- loads ground truth labels + poses (e.g., `data/test_split_labels*.csv`)
- loads one or more prediction CSVs (cluster labels + poses)
- computes the **IMC25 score**: pose-aware mAA + clustering score + harmonic mean
- exports CSV + LaTeX tables for your paper

Where to put predictions:
- Place per-method predictions under `outputs/paper_predictions/<method_name>/submission.csv` (or `submission_*.csv`).
- The notebook auto-discovers these files.


In [1]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Optional, Tuple

import numpy as np
import pandas as pd

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)


def find_repo_root(start: Path | None = None) -> Path:
    start = start or Path.cwd()
    for p in [start] + list(start.parents):
        if (p / "pyproject.toml").exists() or (p / "PIPELINE.md").exists():
            return p
    return start


REPO_ROOT = find_repo_root()
print("REPO_ROOT:", REPO_ROOT)


REPO_ROOT: /home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025


In [2]:
# --- Scoring implementation (matches notebooks/08_pose_scoring_metrics.ipynb) ---

IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".webp"}


def parse_floats_semicolon(s: str, expected: int) -> Optional[np.ndarray]:
    if not isinstance(s, str):
        return None
    s = s.strip()
    if not s or s.lower() == "nan":
        return None
    parts = s.split(";")
    if len(parts) != expected:
        return None
    arr = np.array([float(x) for x in parts], dtype=np.float64)
    if np.any(~np.isfinite(arr)):
        return None
    return arr


def parse_rotation(s: str) -> Optional[np.ndarray]:
    v = parse_floats_semicolon(s, 9)
    if v is None:
        return None
    return v.reshape(3, 3)


def parse_translation(s: str) -> Optional[np.ndarray]:
    return parse_floats_semicolon(s, 3)


def camera_center(R: np.ndarray, t: np.ndarray) -> np.ndarray:
    # COLMAP convention: x_cam = R * x_world + t
    return -R.T @ t


def harmonic_mean(a: float, b: float) -> float:
    if a <= 0 or b <= 0:
        return 0.0
    return 2.0 / (1.0 / a + 1.0 / b)


def normalize_image_key(dataset: str, *, image_id: Optional[str] = None, image: Optional[str] = None) -> str:
    # Derive a stable image key (stem) from either `image` or `image_id`.
    if isinstance(image, str) and image.strip() and image.lower() != "nan":
        return Path(image).stem
    if not isinstance(image_id, str) or not image_id.strip():
        raise ValueError("Need either `image` or `image_id` to derive an image key")

    s = image_id.strip()
    if s.endswith("_public") or s.endswith("_private"):
        s = s.rsplit("_", 1)[0]
    prefix = f"{dataset}_"
    if s.startswith(prefix):
        s = s[len(prefix) :]
    return Path(s).stem


def umeyama_similarity(X: np.ndarray, Y: np.ndarray, *, with_scale: bool = True) -> Tuple[float, np.ndarray, np.ndarray]:
    # Estimate similarity transform s,R,t such that Y pprox s*R*X + t (Umeyama 1991).
    if X.shape != Y.shape or X.ndim != 2 or X.shape[1] != 3:
        raise ValueError("X and Y must have shape (N,3) and match")
    n = X.shape[0]
    if n < 3:
        raise ValueError("Need at least 3 points for a stable 3D similarity")

    muX = X.mean(axis=0)
    muY = Y.mean(axis=0)
    Xc = X - muX
    Yc = Y - muY

    cov = (Yc.T @ Xc) / n
    U, D, Vt = np.linalg.svd(cov)
    S = np.eye(3)
    if np.linalg.det(U) * np.linalg.det(Vt) < 0:
        S[-1, -1] = -1
    R = U @ S @ Vt

    if with_scale:
        varX = (Xc**2).sum(axis=1).mean()
        scale = float(np.trace(np.diag(D) @ S) / varX) if varX > 0 else 1.0
    else:
        scale = 1.0

    t = muY - scale * (R @ muX)
    return scale, R, t


def estimate_similarity_ransac(
    X: np.ndarray,
    Y: np.ndarray,
    *,
    max_iters: int = 512,
    inlier_threshold: float,
    seed: int = 0,
) -> Optional[Tuple[float, np.ndarray, np.ndarray, np.ndarray]]:
    # RANSAC over 3-point Umeyama fits. Returns (s,R,t,inlier_mask).
    if X.shape[0] < 3:
        return None

    rng = np.random.default_rng(seed)
    n = X.shape[0]

    best = None
    best_inliers = -1
    best_median = np.inf

    idx = np.arange(n)
    for _ in range(max_iters):
        sample = rng.choice(idx, size=3, replace=False)
        try:
            s, R, t = umeyama_similarity(X[sample], Y[sample], with_scale=True)
        except Exception:
            continue
        X_aligned = (s * (R @ X.T)).T + t
        err = np.linalg.norm(Y - X_aligned, axis=1)
        inliers = err < inlier_threshold
        cnt = int(inliers.sum())
        med = float(np.median(err[inliers])) if cnt > 0 else np.inf
        if (cnt > best_inliers) or (cnt == best_inliers and med < best_median):
            best_inliers = cnt
            best_median = med
            best = (s, R, t, inliers)

    if best is None or best_inliers < 3:
        return None

    s0, R0, t0, inliers = best
    try:
        s, R, t = umeyama_similarity(X[inliers], Y[inliers], with_scale=True)
    except Exception:
        s, R, t = s0, R0, t0

    return s, R, t, inliers


def _image_keys_in_dir(dataset_dir: Path) -> set[str]:
    if not dataset_dir.exists():
        return set()
    keys: set[str] = set()
    for p in dataset_dir.iterdir():
        if not p.is_file():
            continue
        if p.suffix.lower() not in IMAGE_EXTS:
            continue
        keys.add(p.stem)
    return keys


def load_ground_truth(
    *,
    labels_csv: Path,
    thresholds_csv: Path,
    data_root: Path,
) -> Tuple[
    Dict[str, Dict[str, set[str]]],
    Dict[Tuple[str, str], np.ndarray],
    Dict[Tuple[str, str], np.ndarray],
    Dict[str, set[str]],
]:
    # Load GT scenes, GT camera centers, thresholds, and all images per dataset.
    gt = pd.read_csv(labels_csv)
    required = {"dataset", "scene", "image", "rotation_matrix", "translation_vector"}
    missing = required - set(gt.columns)
    if missing:
        raise ValueError(f"Ground truth missing columns: {sorted(missing)}")

    gt = gt[list(required)].copy()
    gt["image_key"] = gt["image"].map(lambda x: Path(str(x)).stem)

    scenes: Dict[str, Dict[str, set[str]]] = {}
    centers: Dict[Tuple[str, str], np.ndarray] = {}
    for _, r in gt.iterrows():
        ds = str(r["dataset"])
        sc = str(r["scene"])
        k = str(r["image_key"])
        scenes.setdefault(ds, {}).setdefault(sc, set()).add(k)

        R = parse_rotation(str(r["rotation_matrix"]))
        t = parse_translation(str(r["translation_vector"]))
        if R is None or t is None:
            continue
        centers[(ds, k)] = camera_center(R, t)

    thr = pd.read_csv(thresholds_csv)
    if not {"dataset", "scene", "thresholds"}.issubset(thr.columns):
        raise ValueError("thresholds CSV must have columns: dataset, scene, thresholds")
    thresholds: Dict[Tuple[str, str], np.ndarray] = {}
    for _, r in thr.iterrows():
        ds = str(r["dataset"])
        sc = str(r["scene"])
        parts = [float(x) for x in str(r["thresholds"]).split(";")]
        parts = [x for x in parts if x > 0]
        thresholds[(ds, sc)] = np.array(parts, dtype=np.float64)

    all_images: Dict[str, set[str]] = {}
    for dataset in sorted(scenes.keys()):
        keys = _image_keys_in_dir(data_root / dataset)
        if not keys:
            keys = set(gt.loc[gt["dataset"] == dataset, "image_key"].tolist())
        all_images[dataset] = keys

    return scenes, centers, thresholds, all_images


def load_predictions_with_poses(
    *,
    pred_csv: Path,
    expected_images: Dict[str, set[str]],
    outliers_label: str = "outliers",
) -> Tuple[
    Dict[str, Dict[str, set[str]]],
    Dict[Tuple[str, str], Optional[np.ndarray]],
]:
    df = pd.read_csv(pred_csv)
    cols = set(df.columns)
    if "dataset" not in cols:
        raise ValueError("Predictions CSV must contain a `dataset` column")

    cluster_col = None
    for c in ["scene", "cluster", "cluster_label"]:
        if c in cols:
            cluster_col = c
            break
    if cluster_col is None:
        raise ValueError("Predictions CSV must contain one of: scene / cluster / cluster_label")

    image_col = "image" if "image" in cols else None
    image_id_col = "image_id" if "image_id" in cols else None
    if image_col is None and image_id_col is None:
        raise ValueError("Predictions CSV must contain `image` or `image_id`")

    has_pose = ("rotation_matrix" in cols) and ("translation_vector" in cols)

    keep_cols = ["dataset", cluster_col]
    if image_col:
        keep_cols.append(image_col)
    if image_id_col:
        keep_cols.append(image_id_col)
    if has_pose:
        keep_cols += ["rotation_matrix", "translation_vector"]

    df = df[keep_cols].copy().rename(columns={cluster_col: "pred_cluster"})

    def _row_to_key(r) -> str:
        return normalize_image_key(
            str(r["dataset"]),
            image_id=str(r[image_id_col]) if image_id_col and pd.notna(r[image_id_col]) else None,
            image=str(r[image_col]) if image_col and pd.notna(r[image_col]) else None,
        )

    df["image_key"] = df.apply(_row_to_key, axis=1)

    # Keep only images we expect.
    kept = []
    for dataset, g in df.groupby("dataset", sort=False):
        exp = expected_images.get(str(dataset), set())
        if exp:
            kept.append(g[g["image_key"].isin(exp)])
        else:
            kept.append(g)
    df = pd.concat(kept, ignore_index=True) if kept else df

    # If predictions do not cover all images, treat missing ones as predicted outliers.
    rows_missing = []
    for dataset, exp in expected_images.items():
        present = set(df.loc[df["dataset"] == dataset, "image_key"].tolist())
        missing = sorted(exp - present)
        if not missing:
            continue
        rows_missing.extend({"dataset": dataset, "pred_cluster": outliers_label, "image_key": k} for k in missing)
    if rows_missing:
        df = pd.concat([df, pd.DataFrame(rows_missing)], ignore_index=True)

    clusters: Dict[str, Dict[str, set[str]]] = {}
    for (dataset, cluster), g in df.groupby(["dataset", "pred_cluster"], sort=True):
        clusters.setdefault(str(dataset), {})[str(cluster)] = set(g["image_key"].tolist())

    pred_centers: Dict[Tuple[str, str], Optional[np.ndarray]] = {}
    if has_pose:
        for _, r in df.iterrows():
            ds = str(r["dataset"])
            k = str(r["image_key"])
            R = parse_rotation(str(r.get("rotation_matrix", "")))
            t = parse_translation(str(r.get("translation_vector", "")))
            if R is None or t is None:
                pred_centers[(ds, k)] = None
            else:
                pred_centers[(ds, k)] = camera_center(R, t)
    else:
        for _, r in df.iterrows():
            ds = str(r["dataset"])
            k = str(r["image_key"])
            pred_centers[(ds, k)] = None

    return clusters, pred_centers


def pose_mAA_for_scene_and_cluster(
    *,
    dataset: str,
    scene: str,
    scene_images: set[str],
    cluster_images: set[str],
    gt_centers: Dict[Tuple[str, str], np.ndarray],
    pred_centers: Dict[Tuple[str, str], Optional[np.ndarray]],
    thresholds: np.ndarray,
    min_align_points: int = 3,
    use_ransac: bool = True,
    ransac_iters: int = 512,
) -> Tuple[float, int, int]:
    inter = scene_images & cluster_images
    if not inter:
        return 0.0, 0, len(scene_images)

    X = []
    Y = []
    for k in inter:
        pc = pred_centers.get((dataset, k))
        gc = gt_centers.get((dataset, k))
        if pc is None or gc is None:
            continue
        X.append(pc)
        Y.append(gc)

    if len(X) < min_align_points:
        return 0.0, len(X), len(scene_images)

    X = np.stack(X, axis=0)
    Y = np.stack(Y, axis=0)

    max_thr = float(np.max(thresholds)) if thresholds.size else 0.0
    if max_thr <= 0:
        return 0.0, len(X), len(scene_images)

    if use_ransac:
        est = estimate_similarity_ransac(X, Y, max_iters=ransac_iters, inlier_threshold=max_thr, seed=0)
        if est is None:
            return 0.0, len(X), len(scene_images)
        s, R, t, _inliers = est
    else:
        s, R, t = umeyama_similarity(X, Y, with_scale=True)

    errors = np.full((len(scene_images),), np.inf, dtype=np.float64)
    scene_list = sorted(scene_images)
    for i, k in enumerate(scene_list):
        if k not in cluster_images:
            continue
        pc = pred_centers.get((dataset, k))
        gc = gt_centers.get((dataset, k))
        if pc is None or gc is None:
            continue
        pa = (s * (R @ pc)) + t
        errors[i] = float(np.linalg.norm(gc - pa))

    accs = [(errors < thr).mean() for thr in thresholds]
    return float(np.mean(accs)), len(X), len(scene_images)


@dataclass(frozen=True)
class DatasetScore:
    dataset: str
    pose_mAA: float
    clustering_score: float
    final_score: float
    n_scenes: int
    n_images_in_scenes: int


def score_dataset_pose_aware(
    *,
    dataset: str,
    gt_scenes: Dict[str, set[str]],
    pred_clusters: Dict[str, set[str]],
    gt_centers: Dict[Tuple[str, str], np.ndarray],
    pred_centers: Dict[Tuple[str, str], Optional[np.ndarray]],
    thresholds_by_scene: Dict[Tuple[str, str], np.ndarray],
    outliers_label: str = "outliers",
    min_align_points: int = 3,
    use_ransac: bool = True,
) -> Tuple[DatasetScore, pd.DataFrame]:
    candidates = {k: v for k, v in pred_clusters.items() if k != outliers_label and len(v) > 0}

    rows = []
    total_scene_images = 0
    total_correct = 0
    total_cluster_images = 0
    pose_mAA_weighted_sum = 0.0

    for scene, scene_imgs in sorted(gt_scenes.items()):
        n_scene = len(scene_imgs)
        total_scene_images += n_scene
        thresholds = thresholds_by_scene.get((dataset, scene), np.array([], dtype=np.float64))

        best_cluster = None
        best_pose_maa = -1.0
        best_cs = -1.0
        best_correct = 0
        best_cluster_size = 0
        best_align_pts = 0

        for c_label, c_imgs in candidates.items():
            inter = len(scene_imgs & c_imgs)
            if inter == 0:
                continue
            cs = inter / len(c_imgs) if c_imgs else 0.0
            pose_maa, n_align, _ = pose_mAA_for_scene_and_cluster(
                dataset=dataset,
                scene=scene,
                scene_images=scene_imgs,
                cluster_images=c_imgs,
                gt_centers=gt_centers,
                pred_centers=pred_centers,
                thresholds=thresholds,
                min_align_points=min_align_points,
                use_ransac=use_ransac,
            )
            if (pose_maa > best_pose_maa) or (pose_maa == best_pose_maa and cs > best_cs):
                best_cluster = c_label
                best_pose_maa = pose_maa
                best_cs = cs
                best_correct = inter
                best_cluster_size = len(c_imgs)
                best_align_pts = n_align

        if best_cluster is None:
            best_cluster = "(none)"
            best_pose_maa = 0.0
            best_cs = 0.0
            best_correct = 0
            best_cluster_size = 0
            best_align_pts = 0

        pose_mAA_weighted_sum += best_pose_maa * n_scene
        total_correct += best_correct
        total_cluster_images += best_cluster_size

        rows.append(
            {
                "dataset": dataset,
                "gt_scene": scene,
                "assigned_cluster": best_cluster,
                "pose_mAA(scene)": best_pose_maa,
                "CS(scene)": best_cs,
                "n_scene": n_scene,
                "n_cluster": best_cluster_size,
                "n_correct": best_correct,
                "n_align_pts": best_align_pts,
            }
        )

    pose_mAA = (pose_mAA_weighted_sum / total_scene_images) if total_scene_images else 0.0
    cs = (total_correct / total_cluster_images) if total_cluster_images else 0.0
    final = harmonic_mean(pose_mAA, cs)

    return (
        DatasetScore(
            dataset=dataset,
            pose_mAA=float(pose_mAA),
            clustering_score=float(cs),
            final_score=float(final),
            n_scenes=len(gt_scenes),
            n_images_in_scenes=int(total_scene_images),
        ),
        pd.DataFrame(rows),
    )


def score_methods(
    *,
    gt_labels_csv: Path,
    gt_thresholds_csv: Path,
    gt_data_root: Path,
    methods: list[dict],
    outliers_label: str = "outliers",
    min_align_points: int = 3,
    use_ransac: bool = True,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    gt_scenes_by_dataset, gt_centers, thresholds_by_scene, all_images_by_dataset = load_ground_truth(
        labels_csv=gt_labels_csv,
        thresholds_csv=gt_thresholds_csv,
        data_root=gt_data_root,
    )

    datasets = sorted(gt_scenes_by_dataset.keys())

    per_dataset_rows = []
    per_scene_rows = []

    for m in methods:
        name = str(m["method"])
        pred_csv = Path(m["pred_csv"])
        if not pred_csv.exists():
            print(f"[skip] {name}: missing {pred_csv}")
            continue

        pred_clusters_by_dataset, pred_centers = load_predictions_with_poses(
            pred_csv=pred_csv,
            expected_images=all_images_by_dataset,
            outliers_label=outliers_label,
        )

        for ds in datasets:
            if ds not in pred_clusters_by_dataset:
                continue
            ds_score, per_scene = score_dataset_pose_aware(
                dataset=ds,
                gt_scenes=gt_scenes_by_dataset[ds],
                pred_clusters=pred_clusters_by_dataset[ds],
                gt_centers=gt_centers,
                pred_centers=pred_centers,
                thresholds_by_scene=thresholds_by_scene,
                outliers_label=outliers_label,
                min_align_points=min_align_points,
                use_ransac=use_ransac,
            )
            row = ds_score.__dict__.copy()
            row["method"] = name
            row["pred_csv"] = str(pred_csv)
            per_dataset_rows.append(row)

            per_scene = per_scene.copy()
            per_scene["method"] = name
            per_scene["pred_csv"] = str(pred_csv)
            per_scene_rows.append(per_scene)

    per_dataset_df = pd.DataFrame(per_dataset_rows)
    per_scene_df = pd.concat(per_scene_rows, ignore_index=True) if per_scene_rows else pd.DataFrame()

    if len(per_dataset_df) == 0:
        return per_dataset_df, per_scene_df, pd.DataFrame()

    mean_df = (
        per_dataset_df.groupby("method", as_index=False)
        .agg(
            pose_mAA=("pose_mAA", "mean"),
            clustering_score=("clustering_score", "mean"),
            final_score=("final_score", "mean"),
            n_datasets=("dataset", "nunique"),
        )
        .sort_values("final_score", ascending=False)
    )
    return per_dataset_df, per_scene_df, mean_df


In [3]:
# --- Choose evaluation split (ground truth) ---

# Options available in this repo:
# - 'test_split'
# - 'ets_stairs'
# - 'holdout_ets_sacrecoeur_stpeters'
SPLIT = "test_split"

SPLITS = {
    "test_split": {
        "labels": REPO_ROOT / "data/test_split_labels.csv",
        "images": REPO_ROOT / "data/test_split",
    },
    "ets_stairs": {
        "labels": REPO_ROOT / "data/test_split_labels_ets_stairs.csv",
        "images": REPO_ROOT / "data/test_split_ets_stairs",
    },
    "holdout_ets_sacrecoeur_stpeters": {
        "labels": REPO_ROOT / "data/test_split_labels_holdout_ets_sacrecoeur_stpeters.csv",
        "images": REPO_ROOT / "data/test_split_holdout_ets_sacrecoeur_stpeters",
    },
}

GT_LABELS_CSV = SPLITS[SPLIT]["labels"]
GT_DATA_ROOT = SPLITS[SPLIT]["images"]


# If the split images folder is empty/missing, fall back to a convenient eval root.
# This repo often keeps split *labels* in `data/test_split_labels*.csv` but does not always
# materialize the corresponding `data/test_split*` image folders.
if GT_DATA_ROOT.exists() and not any(p.is_dir() for p in GT_DATA_ROOT.iterdir()):
    fallback = REPO_ROOT / "data/eval_test_split"
    if fallback.exists():
        print(f"[warn] {GT_DATA_ROOT} has no dataset subfolders; using {fallback} (symlinks into data/train)")
        GT_DATA_ROOT = fallback

# Thresholds are defined per (dataset, scene). For offline splits derived from train, reuse train thresholds.
GT_THRESHOLDS_CSV = REPO_ROOT / "data/train_thresholds.csv"

for p in [GT_LABELS_CSV, GT_THRESHOLDS_CSV]:
    if not p.exists():
        raise FileNotFoundError(p)
if not GT_DATA_ROOT.exists():
    raise FileNotFoundError(GT_DATA_ROOT)

print("GT_LABELS_CSV:", GT_LABELS_CSV)
print("GT_DATA_ROOT :", GT_DATA_ROOT)
print("GT_THRESHOLDS:", GT_THRESHOLDS_CSV)

OUTLIERS_LABEL = "outliers"
MIN_ALIGN_POINTS = 3
USE_RANSAC = True


# --- Methods to score ---

# Auto-discover any predictions placed at:
#   outputs/paper_predictions/<method_name>/submission*.csv
PRED_ROOT = REPO_ROOT / "outputs/paper_predictions"
PRED_ROOT.mkdir(parents=True, exist_ok=True)

discovered = []
for pred_csv in sorted(PRED_ROOT.glob("*/submission*.csv")):
    discovered.append({"method": pred_csv.parent.name, "pred_csv": pred_csv})

# Always include an oracle upper bound (GT as predictions).
METHODS = [{"method": "oracle_gt", "pred_csv": GT_LABELS_CSV}] + discovered

print(f"Discovered methods: {len(discovered)}")
pd.DataFrame(METHODS)


GT_LABELS_CSV: /home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/data/test_split_labels.csv
GT_DATA_ROOT : /home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/data/test_split
GT_THRESHOLDS: /home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/data/train_thresholds.csv
Discovered methods: 0


,method,pred_csv
0,oracle_gt,/home/shamika/documents/prog_workspace/ml_imc2...


## (Optional) Generate predictions for built-in baselines

If you want the notebook to *produce* predictions (instead of only scoring existing CSVs), set `RUN_PIPELINES = True` below.

Outputs go to `outputs/paper_predictions/<method_name>/submission.csv`, which will then be auto-discovered by the scoring cells.



In [4]:
import os
import shlex
import subprocess
import sys


def run_cmd(cmd: list[str], *, env: dict[str, str] | None = None) -> None:
    print('+', shlex.join(cmd), flush=True)
    subprocess.run(cmd, check=True, env=env)


RUN_PIPELINES = False  # set True to actually run

# Optional: Hugging Face mirror endpoint for model downloads.
HF_ENDPOINT = os.environ.get('HF_ENDPOINT', None)  # e.g. 'https://hf-mirror.com'

# COLMAP runner for reconstruction.
COLMAP_RUNNER = 'docker'  # 'auto'|'local'|'docker'

# Avoid retraining retrieval inside the pipeline.
# Requires that `outputs/retrieval_finetune/best.pt` already exists.
FORCE_NO_TRAIN = True

# Define baselines you want to generate.
PIPELINE_JOBS = [
    {
        'name': 'ours_colmap_sift_sparse',
        'matcher': 'colmap',
        'dense': False,
        'depth_fallback': False,
    },
    {
        'name': 'ours_auto_dense_depth',
        'matcher': 'auto',
        'dense': True,
        'depth_fallback': True,
    },
]

if RUN_PIPELINES:
    env = os.environ.copy()
    if HF_ENDPOINT:
        env['HF_ENDPOINT'] = str(HF_ENDPOINT)

    for job in PIPELINE_JOBS:
        out_dir = PRED_ROOT / job['name']
        out_dir.mkdir(parents=True, exist_ok=True)

        cmd = [
            sys.executable,
            'scripts/run_pipeline.py',
            '--data-test-root',
            str(GT_DATA_ROOT),
            '--cache-root',
            str(out_dir / 'cache_retrieval'),
            '--clusters-csv',
            str(out_dir / 'clusters_retrieval.csv'),
            '--recon-output-root',
            str(out_dir / 'recon'),
            '--submission-csv',
            str(out_dir / 'submission.csv'),
            '--runner',
            str(COLMAP_RUNNER),
            '--matcher',
            str(job['matcher']),
            '--overwrite',
        ]

        if FORCE_NO_TRAIN:
            cmd += ['--no-train']

        if job.get('dense', False):
            cmd += ['--dense']

        if job.get('depth_fallback', False):
            cmd += ['--depth-fallback', '--dense-min-vertices', '5000']

        if HF_ENDPOINT:
            cmd += ['--hf-endpoint', str(HF_ENDPOINT)]

        run_cmd(cmd, env=env)


In [5]:
# --- Score all methods and export paper tables ---

per_dataset_df, per_scene_df, mean_df = score_methods(
    gt_labels_csv=GT_LABELS_CSV,
    gt_thresholds_csv=GT_THRESHOLDS_CSV,
    gt_data_root=GT_DATA_ROOT,
    methods=METHODS,
    outliers_label=OUTLIERS_LABEL,
    min_align_points=MIN_ALIGN_POINTS,
    use_ransac=USE_RANSAC,
)

display(mean_df)

OUT_DIR = REPO_ROOT / "outputs/paper_tables" / SPLIT
OUT_DIR.mkdir(parents=True, exist_ok=True)

if len(per_dataset_df) > 0:
    per_dataset_df.to_csv(OUT_DIR / "sota_per_dataset.csv", index=False)
if len(per_scene_df) > 0:
    per_scene_df.to_csv(OUT_DIR / "sota_per_scene.csv", index=False)
if len(mean_df) > 0:
    mean_df.to_csv(OUT_DIR / "sota_mean.csv", index=False)

    # LaTeX table (simple). You can post-process for styling.
    latex_df = mean_df[["method", "final_score", "pose_mAA", "clustering_score", "n_datasets"]].copy()
    latex = latex_df.to_latex(index=False, float_format=lambda x: f"{x:.4f}")
    (OUT_DIR / "sota_mean.tex").write_text(latex)
    print(latex)

print("[ok] wrote tables to", OUT_DIR)


,method,pose_mAA,clustering_score,final_score,n_datasets
0,oracle_gt,0.954545,1.0,0.97561,3


\begin{tabular}{lrrrr}
\toprule
method & final_score & pose_mAA & clustering_score & n_datasets \\
\midrule
oracle_gt & 0.9756 & 0.9545 & 1.0000 & 3 \\
\bottomrule
\end{tabular}

[ok] wrote tables to /home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025/outputs/paper_tables/test_split
